In [4]:
import sys                                                                                                                                                                                 
import requests                                                                                                                                                                            
import pandas as pd                                                                                                                                                                        
import duckdb                                                                                                                                                                              
import time                                                                                                                                                                                
from pathlib import Path                                                                                                                                                                   
from datetime import datetime                                                                                                                                                              
from dateutil.relativedelta import relativedelta                                                                                                                                           
                                                                                                                                                                                            
DATA_DIR = Path("/Users/elenamurray/Documents/Documents/Repositories/MDS_THESIS/Master-Thesis-2026/data")                                                                                  
                                                                                                                                                                                            
API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"                                                                                                                                               
headers = {"Ocp-Apim-Subscription-Key": API_KEY}                                                                                                                                       
                                                                                                                                                                                            
con = duckdb.connect(str(DATA_DIR / "processed" / "usa_data" / "pjm_lmp_test.duckdb"))                                                                                                     
con.execute("""                                                                                                                                                                            
    CREATE TABLE IF NOT EXISTS rt_hrl_lmps (                                                                                                                                               
        datetime_beginning_utc VARCHAR,                                                                                                                                                    
        datetime_beginning_ept VARCHAR,                                                                                                                                                    
        pnode_id BIGINT,                                                                                                                                                                   
        pnode_name VARCHAR,                                                                                                                                                                
        voltage VARCHAR,                                                                                                                                                                   
        equipment VARCHAR,                                                                                                                                                                 
        type VARCHAR,                                                                                                                                                                      
        zone VARCHAR,                                                                                                                                                                      
        system_energy_price_rt DOUBLE,                                                                                                                                                     
        total_lmp_rt DOUBLE,                                                                                                                                                               
        congestion_price_rt DOUBLE,                                                                                                                                                        
        marginal_loss_price_rt DOUBLE,                                                                                                                                                     
        row_is_current BOOLEAN,                                                                                                                                                            
        version_nbr INTEGER                                                                                                                                                                
    )                                                                                                                                                                                      
""")                                                                                                                                                                                       
                                                                                                                                                                                            
def download_month(start_date, node_type):                                                                                                                                                 
    end_date = start_date + relativedelta(months=1) - relativedelta(days=1)
    date_str = f"{start_date.month}-{start_date.day}-{start_date.year} 00:00 to {end_date.month}-{end_date.day}-{end_date.year} 23:00"                                                     
                                                                                                                                                                                            
    all_rows = []                                                                                                                                                                          
    start_row = 1                                                                                                                                                                          
    while True:                                                                                                                                                                            
        r = requests.get(                                                                                                                                                                  
            "https://api.pjm.com/api/v1/rt_hrl_lmps",                                                                                                                                      
            headers=headers,                                                                                                                                                               
            params={                                                                                                                                                                       
                "rowCount": 50000,                                                                                                                                                         
                "startRow": start_row,                                                                                                                                                     
                "datetime_beginning_ept": date_str,                                                                                                                                        
                "row_is_current": 1,                                                                                                                                                       
                "type": node_type,                                                                                                                                                         
            }                                                                                                                                                                              
        )                                                                                                                                                                                  
        items = r.json().get("items", [])                                                                                                                                                  
        all_rows.extend(items)                                                                                                                                                             
        print(f"  {node_type} {start_date.strftime('%Y-%m')} — {len(all_rows)} rows so far")                                                                                               
        if len(items) < 50000:                                                                                                                                                             
            break                                                                                                                                                                          
        start_row += 50000                                                                                                                                                                 
        time.sleep(10)                                                                                                                                                                    
                                                                                                                                                                                            
    return pd.DataFrame(all_rows)                                                                                                                                                          
                                                                                                                                                                                            
# Test: all months of 2023                                                                                                                                                                 
start = datetime(2023, 1, 1)
total_rows = 0                                                                                                                                                                             
                                                                                                                                                                                            
for month in range(12):                                                                                                                                                                    
    current = start + relativedelta(months=month)                                                                                                                                          
    month_start = time.time()                                                                                                                                                              
                                                                                                                                                                                            
    for node_type in ["LOAD", "GEN"]:                                                                                                                                                      
        df = download_month(current, node_type)                                                                                                                                            
        if not df.empty:                                                                                                                                                                   
            con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")                                                                                                                        
            total_rows += len(df)                                                                                                                                                          
                                                                                                                                                                                            
    elapsed = time.time() - month_start                                                                                                                                                    
    print(f"Month {current.strftime('%Y-%m')} done in {elapsed:.0f}s — total rows so far: {total_rows:,}")                                                                                 
                                                                                                                                                                                            
con.close()                                                                                                                                                                                
print(f"\nDone. Total rows: {total_rows:,}")                

  LOAD 2023-01 — 50000 rows so far
  LOAD 2023-01 — 100000 rows so far
  LOAD 2023-01 — 150000 rows so far
  LOAD 2023-01 — 200000 rows so far
  LOAD 2023-01 — 250000 rows so far
  LOAD 2023-01 — 300000 rows so far
  LOAD 2023-01 — 350000 rows so far
  LOAD 2023-01 — 400000 rows so far
  LOAD 2023-01 — 450000 rows so far
  LOAD 2023-01 — 500000 rows so far
  LOAD 2023-01 — 550000 rows so far
  LOAD 2023-01 — 600000 rows so far
  LOAD 2023-01 — 650000 rows so far
  LOAD 2023-01 — 700000 rows so far
  LOAD 2023-01 — 750000 rows so far
  LOAD 2023-01 — 800000 rows so far
  LOAD 2023-01 — 850000 rows so far
  LOAD 2023-01 — 900000 rows so far
  LOAD 2023-01 — 950000 rows so far
  LOAD 2023-01 — 1000000 rows so far
  LOAD 2023-01 — 1050000 rows so far
  LOAD 2023-01 — 1100000 rows so far
  LOAD 2023-01 — 1150000 rows so far
  LOAD 2023-01 — 1200000 rows so far
  LOAD 2023-01 — 1250000 rows so far
  LOAD 2023-01 — 1300000 rows so far
  LOAD 2023-01 — 1350000 rows so far
  LOAD 2023-01 — 1400

KeyboardInterrupt: 

In [10]:
list(Path(DATA_DIR / "processed" / "usa_data").glob("*.parquet")) 

[]